# Router Stage 1 (Gatekeeper) — v2 Training

## Why this notebook exists

The v1 gatekeeper (`router_stage1_efficientnet_b0.pt`) reported 99.99% test accuracy, but that
number was measured on a held-out split of the **same single negative source it trained on:
CIFAR-10**. In production testing via the FastAPI backend, it confidently (>99.9%) misclassified
ordinary photos (an apple, a bird) as "medical" — including the bird, which *is* a CIFAR-10 class,
proving the model memorized "CIFAR-10-ness" rather than learning a general "is this a medical
scan" boundary.

## What's different this time

1. **Negative class is now built from 5 independent, diverse real-world photo sources** (not one
   32×32 toy dataset), plus a synthetic hard-negative category for screenshots/documents.
2. **Positive class is resampled in a balanced way** from all 7 existing specialist datasets
   already on disk, so no single modality (e.g. the 277k-patch breast histology set) dominates.
3. **Evaluation reports per-source accuracy**, not just one aggregate number — so a repeat of the
   "99.99% but only on CIFAR" failure would be caught before the model ever reaches the backend.
4. **A genuine generalization test**: one entire negative source is held out of training entirely
   and only ever seen at final evaluation, plus there's a slot for the user to drop in their own
   real photos (the exact kind of test that caught the v1 bug) for a final gut check.
5. **Checkpoint saves `class_names` properly this time** — the v1 pneumonia/kidney/brain/retinal
   bug (no class labels saved, silently falling back to `class_0`/`class_1`...) doesn't apply here
   since this is binary and fixed-name, but the checkpoint format now matches what
   `backend/app/inference/model_loader.py` expects out of the box (`model_state_dict`,
   `class_names`, plus a redundant `classes` key for extra safety).
6. **Same EfficientNet-B0 "wide head" architecture** used by the other 3 specialists
   (`Dropout→Linear(1280,512)→ReLU→Dropout→Linear(512,num_classes)`) — this checkpoint will load
   through the existing backend loader with zero code changes.

## Output

`models/router/router_stage1_v2_efficientnet_b0.pt` — deliberately a **new filename**, not an
overwrite. Swap it in via `backend/app/config.py`'s `ROUTER_STAGE1_PATH` only after you've
confirmed the per-source and generalization-set accuracy look good. Keep the v1 checkpoint around
until then.

## Step 0 — Environment & GPU check

In [ ]:
import os, sys, time, json, random, shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")

torch.backends.cudnn.benchmark = True  # established perf fix from prior specialist training

## Step 1 — Config

`INCLUDE_SUN397` defaults to `False`. SUN397 is excellent (108,754 images, 397 diverse scene
categories) but its tarball is ~36 GB — if you have the disk space and want maximum diversity,
flip it to `True`. The default five sources (Food101, Places365-val, OxfordIIITPet, Country211,
plus synthetic screenshots) total roughly 8-10 GB and already provide far more diversity than
CIFAR-10 alone did.

In [ ]:
PROJECT_ROOT = Path(r"E:\\programming\\CliniqAI")
BACKEND_ROOT = PROJECT_ROOT / "backend"

DATA_RAW = PROJECT_ROOT / "data" / "raw"
NEG_ROOT = PROJECT_ROOT / "data" / "raw" / "gatekeeper_negatives_v2"
NEG_ROOT.mkdir(parents=True, exist_ok=True)

# Existing specialist folders -- source of the POSITIVE (medical) class.
# We only care about "is this a medical image", so we glob every image file
# under each folder regardless of internal class-subfolder structure.
POSITIVE_SOURCES = {
    "brain_mri": DATA_RAW / "brain_tumor",
    "kidney_ct": DATA_RAW / "kidney_stone",
    "retinal": DATA_RAW / "diabetic_retinopathy",
    "skin_lesion": DATA_RAW / "skin_lesion",
    "breast_histology": DATA_RAW / "breast_histology",
    "bone_fracture": DATA_RAW / "bone_fracture",
    "pneumonia": DATA_RAW / "pneumonia",
}

INCLUDE_SUN397 = False  # set True only if you have ~40GB free disk space to spare

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Target images per positive modality and per negative source -- caps at whatever
# is actually available, so this is a ceiling, not a guarantee.
TARGET_PER_POSITIVE_SOURCE = 12000
TARGET_PER_NEGATIVE_SOURCE = 12000
NUM_SYNTHETIC_SCREENSHOTS = 6000

BATCH_SIZE = 64          # safe with AMP on 6GB VRAM (v1 without AMP overflowed at 64 -- AMP roughly halves activation memory)
NUM_WORKERS = 4
EPOCHS = 30
PATIENCE = 7
LR = 1e-3
WEIGHT_DECAY = 1e-4

CHECKPOINT_DIR = PROJECT_ROOT / "models" / "router"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "router_stage1_v2_efficientnet_b0.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Step 2 — Disk space sanity check

In [ ]:
usage = shutil.disk_usage(PROJECT_ROOT.drive + "\\\\")
free_gb = usage.free / 1024**3
estimated_need_gb = 10 + (36 if INCLUDE_SUN397 else 0)  # rough estimate for the download set below

print(f"Free space on {PROJECT_ROOT.drive}: {free_gb:.1f} GB")
print(f"Estimated space needed for this notebook's downloads: ~{estimated_need_gb} GB")
if free_gb < estimated_need_gb + 5:  # +5GB safety margin
    print("\n[WARNING] Free space is close to the estimated requirement. "
          "Consider freeing space or setting INCLUDE_SUN397 = False before continuing.")
else:
    print("\nLooks fine, continuing.")

## Step 3 — Download diverse negative (non-medical) sources

All five direct-download from stable, non-Google-Drive mirrors (Food101 → ETH Zurich,
Places365 → MIT CSAIL, OxfordIIITPet → Oxford, Country211 → OpenAI/Azure CDN, SUN397 →
Princeton) -- no Kaggle auth needed, no flaky Google Drive quota errors.

In [ ]:
food_root = NEG_ROOT / "food101"
food_root.mkdir(exist_ok=True)
print("Downloading Food-101 (~5GB, 101,000 food images)...")
_ = datasets.Food101(root=str(food_root), split="train", download=True)
print("Done.")

In [ ]:
places_root = NEG_ROOT / "places365"
places_root.mkdir(exist_ok=True)
print("Downloading Places365 (val split, small images, ~500MB, 36,500 scene images, 365 categories)...")
_ = datasets.Places365(root=str(places_root), split="val", small=True, download=True)
print("Done.")

In [ ]:
pets_root = NEG_ROOT / "oxford_pets"
pets_root.mkdir(exist_ok=True)
print("Downloading Oxford-IIIT Pet (~800MB, 7,400 cat/dog images)...")
_ = datasets.OxfordIIITPet(root=str(pets_root), download=True)
print("Done.")

In [ ]:
country_root = NEG_ROOT / "country211"
country_root.mkdir(exist_ok=True)
print("Downloading Country211 (~2.6GB, 31,650 real-world street/landscape photos)...")
_ = datasets.Country211(root=str(country_root), split="train", download=True)
print("Done.")

In [ ]:
if INCLUDE_SUN397:
    sun_root = NEG_ROOT / "sun397"
    sun_root.mkdir(exist_ok=True)
    print("Downloading SUN397 (~36GB, 108,754 images, 397 scene/object categories)...")
    _ = datasets.SUN397(root=str(sun_root), download=True)
    print("Done.")
else:
    print("Skipping SUN397 (INCLUDE_SUN397 = False).")

## Step 4 — Generate synthetic "screenshot / document" hard negatives

A category no photo dataset covers well: users accidentally uploading a screenshot, a scanned
document, or a phone photo of text instead of a medical scan. These are rendered locally with
PIL -- randomized fonts, layouts, colors, and fake "UI chrome" -- so there's no download
dependency and no risk of accidentally including copyrighted material.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import random as _rnd

synthetic_root = NEG_ROOT / "synthetic_screenshots"
synthetic_root.mkdir(exist_ok=True)

LOREM_WORDS = ("lorem ipsum dolor sit amet consectetur adipiscing elit sed do eiusmod tempor "
               "incididunt ut labore et dolore magna aliqua enim minim veniam quis nostrud "
               "exercitation ullamco laboris nisi aliquip ex ea commodo consequat duis aute "
               "irure reprehenderit voluptate velit esse cillum eu fugiat nulla pariatur "
               "excepteur sint occaecat cupidatat non proident sunt culpa qui officia deserunt "
               "mollit anim id est laborum").split()

def random_text(n_words):
    return " ".join(_rnd.choice(LOREM_WORDS) for _ in range(n_words))

def make_synthetic_image(size=(600, 800)):
    kind = _rnd.choice(["document", "chat_screenshot", "code_editor", "spreadsheet", "webpage"])
    bg = _rnd.choice(["#ffffff", "#f5f5f5", "#1e1e1e", "#0d1117", "#eef1f5"])
    img = Image.new("RGB", size, bg)
    draw = ImageDraw.Draw(img)
    dark_bg = bg in ("#1e1e1e", "#0d1117")
    text_color = "#e6e6e6" if dark_bg else "#222222"
    accent = _rnd.choice(["#4285f4", "#25d366", "#0078d4", "#ff5722", "#6c5ce7"])

    if kind in ("document", "spreadsheet"):
        y = 40
        while y < size[1] - 40:
            line_words = _rnd.randint(4, 12)
            draw.text((40, y), random_text(line_words), fill=text_color)
            y += _rnd.randint(22, 34)
        if kind == "spreadsheet":
            for gx in range(0, size[0], _rnd.randint(60, 120)):
                draw.line([(gx, 0), (gx, size[1])], fill="#cccccc", width=1)
            for gy in range(0, size[1], _rnd.randint(30, 50)):
                draw.line([(0, gy), (size[0], gy)], fill="#cccccc", width=1)
    elif kind == "chat_screenshot":
        y = 30
        for _ in range(_rnd.randint(6, 14)):
            bubble_w = _rnd.randint(150, 400)
            bubble_h = _rnd.randint(30, 60)
            x = _rnd.choice([30, size[0] - bubble_w - 30])
            color = accent if x == 30 else "#e0e0e0"
            draw.rounded_rectangle([x, y, x + bubble_w, y + bubble_h], radius=12, fill=color)
            y += bubble_h + 15
    elif kind == "code_editor":
        img = Image.new("RGB", size, "#1e1e1e")
        draw = ImageDraw.Draw(img)
        y = 20
        indent_levels = [0, 20, 40, 20, 0]
        for i in range(_rnd.randint(15, 30)):
            indent = _rnd.choice(indent_levels)
            width = _rnd.randint(80, 350)
            color = _rnd.choice(["#569cd6", "#9cdcfe", "#ce9178", "#4ec9b0", "#d4d4d4"])
            draw.rectangle([40 + indent, y, 40 + indent + width, y + 12], fill=color)
            y += 20
    else:  # webpage
        draw.rectangle([0, 0, size[0], 70], fill=accent)
        y = 100
        while y < size[1] - 40:
            draw.rectangle([40, y, size[0] - 40, y + 12], fill="#cccccc" if not dark_bg else "#444444")
            y += 26

    return img

n_existing = len(list(synthetic_root.glob("*.png")))
n_to_make = max(0, NUM_SYNTHETIC_SCREENSHOTS - n_existing)
print(f"Generating {n_to_make} synthetic screenshot/document images...")
for i in range(n_to_make):
    img = make_synthetic_image()
    img.save(synthetic_root / f"synthetic_{i:05d}.png")
    if (i + 1) % 1000 == 0:
        print(f"  {i+1}/{n_to_make}")
print("Done.")

## Step 5 — Build the balanced POSITIVE (medical) image list

Globs every image file under each of the 7 specialist source folders and samples a capped,
balanced amount per modality, so no single modality's dataset size distorts what "medical"
looks like to the gatekeeper.

In [ ]:
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def find_images(root, limit=None):
    if not root.exists():
        return []
    paths = [p for p in root.rglob("*") if p.suffix.lower() in IMG_EXTENSIONS]
    return paths

positive_records = []  # (path, source_tag)
print("Positive (medical) sources:")
for tag, folder in POSITIVE_SOURCES.items():
    all_imgs = find_images(folder)
    if not all_imgs:
        print(f"  [WARNING] {tag}: no images found under {folder} -- skipping. "
              f"Check the folder path if this is unexpected.")
        continue
    n = min(len(all_imgs), TARGET_PER_POSITIVE_SOURCE)
    sampled = random.sample(all_imgs, n)
    positive_records.extend((p, tag) for p in sampled)
    print(f"  {tag:<20} available={len(all_imgs):<7} sampled={n}")

print(f"\nTotal positive images: {len(positive_records)}")

## Step 6 — Build the balanced NEGATIVE (non-medical) image list

In [ ]:
NEGATIVE_SOURCES = {
    "food101": food_root,
    "places365": places_root,
    "oxford_pets": pets_root,
    "country211": country_root,
    "synthetic_screenshots": synthetic_root,
}
if INCLUDE_SUN397:
    NEGATIVE_SOURCES["sun397"] = sun_root

negative_records = []  # (path, source_tag)
print("Negative (non-medical) sources:")
for tag, folder in NEGATIVE_SOURCES.items():
    all_imgs = find_images(folder)
    if not all_imgs:
        print(f"  [WARNING] {tag}: no images found under {folder} -- skipping.")
        continue
    n = min(len(all_imgs), TARGET_PER_NEGATIVE_SOURCE)
    sampled = random.sample(all_imgs, n)
    negative_records.extend((p, tag) for p in sampled)
    print(f"  {tag:<25} available={len(all_imgs):<8} sampled={n}")

print(f"\nTotal negative images: {len(negative_records)}")
print(f"Total positive images: {len(positive_records)}")
print("(A large imbalance here is fine -- the DataLoader/sampler below balances at batch level "
      "if needed; a same-order-of-magnitude match is what actually matters.)")

## Step 7 — Train / val / test split, with a genuine held-out generalization set

Two extra things beyond a normal split:

1. **Stratify by (label, source_tag), not just label** -- so every source appears proportionally
   in train/val/test, rather than e.g. all of Food101 landing in train and none in test.
2. **Hold out one negative source entirely from training** (`country211`, chosen arbitrarily) --
   it is never seen during training, only at final evaluation. If the model does well on it
   anyway, that's real evidence of generalization, not memorization -- exactly the property v1
   was missing.

In [ ]:
HELD_OUT_SOURCE = "country211"  # never included in train/val -- generalization-only

# Split off the held-out generalization source first.
generalization_records = [(p, "negative", tag) for p, tag in negative_records if tag == HELD_OUT_SOURCE]
trainable_negative = [(p, "negative", tag) for p, tag in negative_records if tag != HELD_OUT_SOURCE]
trainable_positive = [(p, "medical", tag) for p, tag in positive_records]

all_trainable = trainable_positive + trainable_negative
random.shuffle(all_trainable)

def stratified_split(records, val_frac=0.15, test_frac=0.15):
    buckets = defaultdict(list)
    for rec in records:
        buckets[(rec[1], rec[2])].append(rec)
    train, val, test = [], [], []
    for key, items in buckets.items():
        random.shuffle(items)
        n = len(items)
        n_val = int(n * val_frac)
        n_test = int(n * test_frac)
        val.extend(items[:n_val])
        test.extend(items[n_val:n_val + n_test])
        train.extend(items[n_val + n_test:])
    random.shuffle(train)
    return train, val, test

train_records, val_records, test_records = stratified_split(all_trainable)

print(f"Train: {len(train_records)}")
print(f"Val:   {len(val_records)}")
print(f"Test:  {len(test_records)}  (in-distribution, held-out split of trained sources)")
print(f"Generalization set: {len(generalization_records)}  (from '{HELD_OUT_SOURCE}', NEVER trained on)")

CLASS_NAMES = ["non_medical", "medical"]
label_to_idx = {"negative": 0, "medical": 1}

## Step 8 — Standalone Dataset module (Windows multiprocessing fix)

Same convention as `scripts/gatekeeper_dataset.py` / `scripts/modality_dataset.py`: the Dataset
class must live in an importable `.py` file, not a notebook cell, or `num_workers>0` silently
falls back to single-process loading on Windows. `ImageFile.LOAD_TRUNCATED_IMAGES = True` is set
at module level so it applies inside worker subprocesses too.

In [ ]:
%%writefile E:\programming\CliniqAI\scripts\gatekeeper_dataset_v2.py
"""
Standalone, importable Dataset for Router Stage 1 v2 training.
Must live in its own module (not a notebook cell) for Windows DataLoader
multiprocessing (num_workers > 0) to work -- worker subprocesses re-import
this module fresh, so ImageFile.LOAD_TRUNCATED_IMAGES must be set here at
module level, not just in the training notebook.
"""
from pathlib import Path
from PIL import Image, ImageFile
from torch.utils.data import Dataset

ImageFile.LOAD_TRUNCATED_IMAGES = True


class GatekeeperDatasetV2(Dataset):
    """records: list of (path, label_str, source_tag). label_str in {'medical','negative'}."""

    def __init__(self, records, label_to_idx, transform=None):
        self.records = records
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def _load(self, idx):
        path, label_str, _tag = self.records[idx]
        img = Image.open(path).convert("RGB")
        img.load()
        label = self.label_to_idx[label_str]
        return img, label

    def __getitem__(self, idx):
        try:
            img, label = self._load(idx)
        except (OSError, Image.DecompressionBombError):
            # Runtime fallback: corrupted/unreadable file -- substitute a random
            # different sample instead of crashing the whole training run.
            fallback_idx = (idx + 1) % len(self.records)
            img, label = self._load(fallback_idx)
        if self.transform:
            img = self.transform(img)
        return img, label


## Step 9 — Transforms & DataLoaders

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.gatekeeper_dataset_v2 import GatekeeperDatasetV2

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = GatekeeperDatasetV2(train_records, label_to_idx, transform=train_transform)
val_ds = GatekeeperDatasetV2(val_records, label_to_idx, transform=eval_transform)
test_ds = GatekeeperDatasetV2(test_records, label_to_idx, transform=eval_transform)
generalization_ds = GatekeeperDatasetV2(generalization_records, label_to_idx, transform=eval_transform)

loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
generalization_loader = DataLoader(generalization_ds, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}  "
      f"Test batches: {len(test_loader)}  Generalization batches: {len(generalization_loader)}")

## Step 10 — Model

Same "wide head" architecture as `brain_tumor_specialist_v1.pt` / `kidney_specialist_v1.pt` /
`retinal_specialist_v1.pt`: `classifier.1` (Linear 1280→512) and `classifier.4` (Linear
512→num_classes). This guarantees the checkpoint loads through the existing
`backend/app/inference/model_loader.py` with zero code changes on the backend side.

In [ ]:
def build_model(num_classes=2, device=DEVICE):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    num_ftrs = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(num_ftrs, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes),
    )
    return model.to(device)

model = build_model(num_classes=2, device=DEVICE)
model = model.to(memory_format=torch.channels_last)  # free perf win on RTX 40-series (Ada)
print(model.classifier)

## Step 11 — Training loop

Mixed precision (`torch.cuda.amp`) is the key change for making full use of the RTX 4050's 6GB:
it roughly halves activation memory, which is what makes batch size 64 safe here (batch 64
overflowed into slow shared memory *without* AMP during earlier router training).

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
scaler = torch.cuda.amp.GradScaler()

def run_epoch(loader, train_mode):
    model.train(train_mode)
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad(), torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total

best_val_loss = float("inf")
patience_counter = 0
history = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc = run_epoch(val_loader, train_mode=False)
    scheduler.step(val_loss)
    elapsed = time.time() - t0

    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                     "val_loss": val_loss, "val_acc": val_acc, "seconds": elapsed})
    print(f"Epoch {epoch:2d}/{EPOCHS}  train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}  ({elapsed:.1f}s)")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": CLASS_NAMES,
            "classes": CLASS_NAMES,  # redundant key, matches the format seen in other checkpoints
            "epoch": epoch,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "task": "Router Stage 1 v2 (Gatekeeper: medical vs non-medical)",
            "negative_sources": list(NEGATIVE_SOURCES.keys()),
            "held_out_generalization_source": HELD_OUT_SOURCE,
        }, CHECKPOINT_PATH)
        print(f"  -> saved new best checkpoint (val_loss={val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping -- no val_loss improvement in {PATIENCE} epochs.")
            break

print("\nTraining complete. Best checkpoint:", CHECKPOINT_PATH)

## Step 12 — Evaluate: overall AND per-source breakdown

This is the check that would have caught the v1 bug before it ever reached the backend: a single
aggregate accuracy number can hide a model that's only good at one source. Reporting per-source
accuracy on the in-distribution test set surfaces that immediately.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

def evaluate_with_breakdown(records, loader_batch_size=BATCH_SIZE):
    ds = GatekeeperDatasetV2(records, label_to_idx, transform=eval_transform)
    loader = DataLoader(ds, batch_size=loader_batch_size, shuffle=False, num_workers=NUM_WORKERS)

    all_preds, all_labels, all_tags = [], [], []
    with torch.no_grad(), torch.cuda.amp.autocast():
        idx = 0
        for images, labels in loader:
            images = images.to(DEVICE).to(memory_format=torch.channels_last)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().numpy()
            batch_size = len(preds)
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.numpy().tolist())
            all_tags.extend(r[2] for r in records[idx:idx + batch_size])
            idx += batch_size

    overall_acc = accuracy_score(all_labels, all_preds)
    print(f"Overall accuracy: {overall_acc:.4f}\n")
    print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(all_labels, all_preds))

    print("\nPer-source accuracy (this is the number that matters most):")
    by_source = defaultdict(lambda: {"correct": 0, "total": 0})
    for p, l, t in zip(all_preds, all_labels, all_tags):
        by_source[t]["total"] += 1
        by_source[t]["correct"] += int(p == l)
    for tag, stats in sorted(by_source.items()):
        acc = stats["correct"] / stats["total"]
        flag = "  <-- LOW, investigate" if acc < 0.90 else ""
        print(f"  {tag:<25} n={stats['total']:<6} acc={acc:.4f}{flag}")

    return overall_acc, by_source

print("=" * 60)
print("IN-DISTRIBUTION TEST SET (held-out split of trained sources)")
print("=" * 60)
test_acc, test_breakdown = evaluate_with_breakdown(test_records)

## Step 13 — The real test: generalization to a never-trained-on source

`country211` was excluded from training entirely. If accuracy here is meaningfully lower than
the in-distribution test accuracy above, that's honest signal about how well this generalizes to
truly novel photos -- exactly the property v1 lacked.

In [ ]:
print("=" * 60)
print(f"GENERALIZATION SET ('{HELD_OUT_SOURCE}', never seen during training)")
print("=" * 60)
gen_acc, gen_breakdown = evaluate_with_breakdown(generalization_records)

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"In-distribution test accuracy:  {test_acc:.4f}")
print(f"Held-out generalization accuracy: {gen_acc:.4f}")
gap = test_acc - gen_acc
if gap > 0.05:
    print(f"\n[NOTE] {gap:.1%} gap between in-distribution and held-out accuracy. "
          "Some generalization gap is normal and expected -- a large gap (>10-15%) would suggest "
          "the model is still leaning on source-specific artifacts rather than general "
          "'medical vs not' features, and would be worth investigating further before trusting "
          "this in production.")
else:
    print(f"\nGap is small ({gap:.1%}) -- good sign that this generalizes beyond its training sources.")

## Step 14 — Optional: your own real-world sanity check

The exact test that caught the v1 bug (an apple photo, a bird photo) wasn't in any dataset split
-- it was just a real photo taken casually. Drop any images you want into
`data/raw/manual_sanity_check/` (create the folder, any filenames, any format) and run this cell.
Nothing here is required for the model to be considered "done" -- it's just a final gut check
before swapping this into the backend.

In [ ]:
sanity_dir = DATA_RAW / "manual_sanity_check"
sanity_images = find_images(sanity_dir) if sanity_dir.exists() else []

if not sanity_images:
    print(f"No images found in {sanity_dir} -- create the folder and drop in some test photos "
          f"if you want to run this check (optional).")
else:
    print(f"Found {len(sanity_images)} manual sanity-check images:\n")
    model.eval()
    with torch.no_grad(), torch.cuda.amp.autocast():
        for path in sanity_images:
            img = Image.open(path).convert("RGB")
            tensor = eval_transform(img).unsqueeze(0).to(DEVICE).to(memory_format=torch.channels_last)
            probs = F.softmax(model(tensor), dim=1)[0].cpu().numpy()
            pred = CLASS_NAMES[int(probs.argmax())]
            print(f"  {path.name:<40} -> {pred:<12} (medical={probs[1]:.4f}, non_medical={probs[0]:.4f})")

## Next steps

1. Review the per-source breakdown above -- every source should comfortably clear ~90%+.
   If one source is notably weak, that points to what kind of images still need more
   representation before this is trustworthy.
2. Review the generalization gap in Step 13. Small gap = good sign.
3. If satisfied, point the backend at the new checkpoint in `backend/app/config.py`:
   ```python
   ROUTER_STAGE1_PATH = MODELS_ROOT / "router" / "router_stage1_v2_efficientnet_b0.pt"
   ```
4. Restart `uvicorn` and re-run the exact apple/bird test that exposed the v1 bug, plus a few
   more real photos, via `/docs`.
5. Keep the v1 checkpoint (`router_stage1_efficientnet_b0.pt`) around until v2 is fully confirmed
   working end-to-end -- don't delete it yet.
6. Commit `scripts/gatekeeper_dataset_v2.py` and this notebook to a new branch off `develop`
   (e.g. `feature/gatekeeper-v2`), following the established branch → PR → develop convention.